In [1]:
from scripts.default_import import *

N_YEARS = 10
YEARS = np.arange(1, N_YEARS + 1)

# Reformat the capacity scenarios

In [2]:
capacity_scenario_names = [
    "base_capacity",
    "IPV_Shortage",
    "pandemic",
    "funding_delay",
    "innacurate_forecast",
    "supply_chain",
    "other",
]
capacity_scenario_probs = {
    "base_capacity": 0.8418,
    "IPV_Shortage": 0.0173,
    "pandemic": 0.0275,
    "funding_delay": 0.0778,
    "innacurate_forecast": 0.0023,
    "supply_chain": 0.0058,
    "other": 0.0275,
}

In [3]:
dfs = []
for scenario in capacity_scenario_names:
    temp = pd.read_excel(
        f"{Github_path}/DATA/production_capacity_scenarios/production_capacity_scenarios.xlsx",
        sheet_name=scenario,
    )
    temp["capacity_scenario"] = scenario
    temp["capacity_scenario_probability"] = capacity_scenario_probs[scenario]
    dfs.append(temp)

# Concatenate all the dataframes
capacity_scenarios = pd.concat(dfs, ignore_index=True)
# rename the columns
capacity_scenarios.rename(columns={"Manufacturer": "manufacturer"}, inplace=True)
# Convert years to columns
capacity_scenarios["capacity"] = capacity_scenarios[YEARS].values.tolist()
# Drop the years columns
capacity_scenarios.drop(YEARS, axis=1, inplace=True)

capacity_scenarios.to_csv("data/production_capacity_scenarios.csv", index=False)

In [29]:
capacity_scenarios

,manufacturer,capacity_scenario,capacity_scenario_probability,capacity
0,AJ_Vaccines,base_capacity,0.1324,"[7711003, 7711003, 7711003, 7711003, 7711003, ..."
1,BB_NCIPD,base_capacity,0.1324,"[39223956, 39223956, 39223956, 39223956, 39223..."
2,Bharat_Biotech,base_capacity,0.1324,"[61029105, 61029105, 61029105, 61029105, 61029..."
3,Bilthoven,base_capacity,0.1324,"[12048153, 12048153, 12048153, 12048153, 12048..."
4,Biological_E,base_capacity,0.1324,"[164885690, 164885690, 164885690, 164885690, 1..."
...,...,...,...,...
100,PT_Bio,other_scenario,0.1339,"[45820610, 45733417, 45911731, 45650707, 43771..."
101,Panacea_Biotec,other_scenario,0.1339,"[10874165, 10874165, 10874165, 10874165, 10781..."
102,Pfizer,other_scenario,0.1339,"[83916974, 83916974, 83916974, 83916974, 79721..."
103,Sanofi,other_scenario,0.1339,"[87429432, 87429432, 82202799, 80966413, 83303..."


# Create scenario pairs

In [4]:
demand_scenarios_df = pd.read_csv(
    "data/10_master_scenarios.csv",
    converters={"demand": pd.eval},
    usecols=["demand", "antigen", "demand_scenario", "demand_scenario_probability"],
)

capacity_scenarios_df = pd.read_csv(
    "data/production_capacity_scenarios.csv",
    converters={"capacity": pd.eval},
)


def generate_pairs(demand: pd.DataFrame, capacity: pd.DataFrame, n_pairs: int = 10, verbose: bool = True):
    demand_dictionary = (
        demand.drop_duplicates(subset=["demand_scenario"])
        .set_index("demand_scenario")["demand_scenario_probability"]
        .to_dict()
    )
    capacity_dictionary = (
        capacity.drop_duplicates(subset=["capacity_scenario"])
        .set_index("capacity_scenario")["capacity_scenario_probability"]
        .to_dict()
    )

    # Extract keys and probabilities
    demand_keys = list(demand_dictionary.keys())
    demand_probabilities = list(demand_dictionary.values())

    capacity_keys = list(capacity_dictionary.keys())
    capacity_probabilities = list(capacity_dictionary.values())

    print(f" Unique demand scenarios: {demand_keys}\n") if verbose else None
    print(f" Unique capacity scenarios: {capacity_keys}") if verbose else None

    # ! This is if you want randomly pick both demand and capacity scenarios
    # # Generate n pairs
    # pairs = set()
    # pair_dfs = []
    # prob_dict = {}
    # pair_idx = 1
    # while len(pairs) < n_pairs:
    #     selected_demand = np.random.choice(demand_keys, p=demand_probabilities)
    #     selected_capacity = np.random.choice(capacity_keys, p=capacity_probabilities)

    #     # Combine the probabilities
    #     selected_prob_demand = demand_dictionary[selected_demand]
    #     selected_prob_capacity = capacity_dictionary[selected_capacity]
    #     combined_prob = selected_prob_demand * selected_prob_capacity
    #     prob_dict[pair_idx] = combined_prob

    #     # Get the selected demand and capacity scenarios
    #     selected_demand_df = demand[demand["demand_scenario"] == selected_demand].copy()
    #     selected_demand_df["type"] = "antigen"
    #     selected_demand_df.drop(columns=["demand_scenario_probability", "demand_scenario"], inplace=True)
    #     selected_demand_df.rename(columns={"antigen": "unit", "demand": "values"}, inplace=True)
    #     selected_capacity_df = capacity[capacity["capacity_scenario"] == selected_capacity].copy()
    #     selected_capacity_df["type"] = "manufacturer"
    #     selected_capacity_df.drop(columns=["capacity_scenario_probability", "capacity_scenario"], inplace=True)
    #     selected_capacity_df.rename(columns={"manufacturer": "unit", "capacity": "values"}, inplace=True)
    #     pair_df = pd.concat([selected_demand_df, selected_capacity_df])

    #     # Add additional information
    #     pair_df["temp_prob"] = combined_prob
    #     pair_df["pair_idx"] = pair_idx
    #     pair_df["pair"] = f"Demand: {selected_demand} - Capacity: {selected_capacity}"
    #     pair_dfs.append(pair_df)
    #     pairs.add((selected_demand, selected_capacity))
    #     pair_idx += 1

    pairs, pair_dfs = [], []
    prob_dict = {}
    pair_idx = 1
    for selected_capacity, selected_prob_capacity in capacity_dictionary.items():
        for n in range(n_pairs):
            selected_demand = np.random.choice(demand_keys, p=demand_probabilities)
            # Combine the probabilities
            selected_prob_demand = demand_dictionary[selected_demand]

            combined_prob = selected_prob_demand * selected_prob_capacity
            prob_dict[pair_idx] = combined_prob

            # Get the selected demand and capacity scenarios
            selected_demand_df = demand[demand["demand_scenario"] == selected_demand].copy()
            selected_demand_df["type"] = "antigen"
            selected_demand_df.drop(columns=["demand_scenario_probability", "demand_scenario"], inplace=True)
            selected_demand_df.rename(columns={"antigen": "unit", "demand": "values"}, inplace=True)

            selected_capacity_df = capacity[capacity["capacity_scenario"] == selected_capacity].copy()
            selected_capacity_df["type"] = "manufacturer"
            selected_capacity_df.drop(columns=["capacity_scenario_probability", "capacity_scenario"], inplace=True)
            selected_capacity_df.rename(columns={"manufacturer": "unit", "capacity": "values"}, inplace=True)
            pair_df = pd.concat([selected_demand_df, selected_capacity_df])

            # Add additional information
            pair_df["temp_prob"] = combined_prob
            pair_df["pair_idx"] = pair_idx
            pair_df["pair"] = f"Demand: {selected_demand} - Capacity: {selected_capacity}"
            pair_dfs.append(pair_df)
            pairs.append((selected_demand, selected_capacity))
            pair_idx += 1

    pair_df = pd.concat(pair_dfs, ignore_index=True)
    # Calculate the sum of all probabilities
    total_sum = sum(prob_dict.values())
    # Scale the probabilities so they sum up to 1
    scaled_probabilities = {k: v / total_sum for k, v in prob_dict.items()}
    pair_df["pair_probability"] = pair_df["pair_idx"].map(scaled_probabilities)
    # Drop the temporary probability column
    pair_df.drop(columns=["temp_prob"], inplace=True)
    return pairs, pair_df


scenario_pairs, pair_df = generate_pairs(demand_scenarios_df, capacity_scenarios_df, n_pairs=3, verbose=True)
# Export to json
pair_df.to_json("data/pair_demand_capacity.json", orient="records", lines=True)
pair_df.to_csv("data/pair_demand_capacity.csv", index=False)
pair_df

 Unique demand scenarios: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

 Unique capacity scenarios: ['base_capacity', 'IPV_Shortage', 'pandemic', 'funding_delay', 'innacurate_forecast', 'supply_chain', 'other']


,unit,values,type,pair_idx,pair,pair_probability
0,Diphtheria,"[641443900, 549169300, 792984600, 795763700, 8...",antigen,1,Demand: 6 - Capacity: base_capacity,0.295220
1,HPV,"[28767500, 35606700, 39994900, 45718800, 55903...",antigen,1,Demand: 6 - Capacity: base_capacity,0.295220
2,Hepatitis_B,"[419505800, 407408700, 381821500, 368643100, 4...",antigen,1,Demand: 6 - Capacity: base_capacity,0.295220
3,Hib,"[288767400, 305847000, 347994400, 338611400, 3...",antigen,1,Demand: 6 - Capacity: base_capacity,0.295220
4,Measles,"[507630600, 425029300, 555256200, 679938800, 5...",antigen,1,Demand: 6 - Capacity: base_capacity,0.295220
...,...,...,...,...,...,...
562,PT_Bio,"[45864272.0, 45911731.0, 45911731.0, 45702427....",manufacturer,21,Demand: 6 - Capacity: other,0.009644
563,Panacea_Biotec,"[10874165.0, 10874165.0, 10874165.0, 10874165....",manufacturer,21,Demand: 6 - Capacity: other,0.009644
564,Pfizer,"[83916974.0, 83916974.0, 83916974.0, 79721125....",manufacturer,21,Demand: 6 - Capacity: other,0.009644
565,Sanofi,"[87429432.0, 87429432.0, 82229737.0, 80966413....",manufacturer,21,Demand: 6 - Capacity: other,0.009644


In [101]:
pair_df.set_index("pair_idx")

,unit,values,type,pair,pair_probability
pair_idx,,,,,
1,Diphtheria,"[650619400, 668568900, 665299400, 940140800, 6...",antigen,Demand: 5 - Capacity: base_capacity,0.365930
1,HPV,"[26509800, 35593800, 41761900, 46914800, 51357...",antigen,Demand: 5 - Capacity: base_capacity,0.365930
1,Hepatitis_B,"[361450400, 362974100, 358595100, 337610100, 3...",antigen,Demand: 5 - Capacity: base_capacity,0.365930
1,Hib,"[316577600, 321711600, 337724800, 323457000, 3...",antigen,Demand: 5 - Capacity: base_capacity,0.365930
1,Measles,"[396235400, 525423200, 507983700, 478856400, 6...",antigen,Demand: 5 - Capacity: base_capacity,0.365930
...,...,...,...,...,...
21,PT_Bio,"[45820610, 45733417, 45911731, 45650707, 43771...",manufacturer,Demand: 10 - Capacity: other_scenario,0.008958
21,Panacea_Biotec,"[10874165, 10874165, 10874165, 10874165, 10781...",manufacturer,Demand: 10 - Capacity: other_scenario,0.008958
21,Pfizer,"[83916974, 83916974, 83916974, 83916974, 79721...",manufacturer,Demand: 10 - Capacity: other_scenario,0.008958


In [102]:
pair_df.loc[1]

unit                                                              HPV
values              [26509800, 35593800, 41761900, 46914800, 51357...
type                                                          antigen
pair_idx                                                            1
pair                              Demand: 5 - Capacity: base_capacity
pair_probability                                              0.36593
Name: 1, dtype: object